# Supervised Fine-Tuning (SFT) — MoE Transformer

Fine-tunes the pretrained causal MoE decoder on a mixed instruction-tuning corpus.

**Key differences from pretraining (`train_moe.ipynb`)**
- Loads `train_tokens.bin` / `train_labels.bin` produced by `sft_prepare.py`
- Labels carry `IGNORE_INDEX = -100` for every prompt/context token → loss is computed **only** on response tokens
- `CrossEntropyLoss(ignore_index=-100)` enforces the masking automatically
- Lower LR, shorter warmup, lighter weight-decay compared to pretraining
- Checkpoint is initialised from the pretrained base model defined in `sft_config.json`

## 0 · Environment

In [1]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256"

# Sanity-check installed versions
!pip show torch accelerate tiktoken datasets

Name: torch
Version: 2.9.1+cu130
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: 
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /root/bharg/SFT_prj/venv/lib/python3.12/site-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas, nvidia-cuda-cupti, nvidia-cuda-nvrtc, nvidia-cuda-runtime, nvidia-cudnn-cu13, nvidia-cufft, nvidia-cufile, nvidia-curand, nvidia-cusolver, nvidia-cusparse, nvidia-cusparselt-cu13, nvidia-nccl-cu13, nvidia-nvjitlink, nvidia-nvshmem-cu13, nvidia-nvtx, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, torchaudio, torchvision
---
Name: accelerate
Version: 1.14.0
Summary: Accelerate
Home-page: https://github.com/huggingface/accelerate
Author: The Hugging Face team
Author-email: transformers@huggingface.co
License: Apache
Location: /root/bharg/SFT_prj/venv/lib/python3.12/site-packages
Requires: huggingface_hub, numpy, packaging, psutil, pyyaml

## 1 · Imports

In [2]:
import gc
import json
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn.functional as F
import torch.multiprocessing as mp
import accelerate
from accelerate import Accelerator
from accelerate.utils import DistributedDataParallelKwargs
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from tqdm.auto import tqdm

from model_moe import MoETransformer

## 2 · Config

In [ ]:
with open("sft_config.json", "r") as f:
    cfg = json.load(f)

# ── model ────────────────────────────────────────────────────────────────────
mc = cfg["model_config"]
vocab_size  = mc["vocab_size"]    # 50304
d_model     = mc["d_model"]       # 384
d_ff        = mc["d_ff"]          # 1024
num_layers  = mc["num_layers"]    # 16
num_experts = mc["num_experts"]   # 16
num_heads   = mc["num_heads"]     # 16
block_size  = mc["block_size"]    # 2048
dropout     = mc["dropout"]       # 0.1

# ── training ─────────────────────────────────────────────────────────────────
tc = cfg["training_config"]
batch_size             = tc["batch_size"]               # 2
gradient_acc_steps     = tc["gradient_accumulation_steps"]  # 4  → effective BS = 8 per GPU
max_iters              = tc["max_iters"]                # 50 000
max_supervised_tokens = tc["max_sup_tokens"]
eval_iters             = tc["eval_iters"]               # 100
log_interval           = tc["log_interval"]             # 500
save_interval          = tc["save_interval"]            # 2 000
lr                     = tc["lr"]                       # 5e-6
min_lr                 = tc["min_lr"]                   # 1e-6
warmup_iters           = tc["warmup_iters"]             # 1 000
weight_decay           = tc["weight_decay"]             # 0.01
beta1                  = tc["beta1"]                    # 0.9
beta2                  = tc["beta2"]                    # 0.95
grad_clip              = tc["grad_clip"]                # 1.0
aux_loss_weight        = tc["aux_loss_weight"]          # 0.001

# ── data ─────────────────────────────────────────────────────────────────────
dc = cfg["data_config"]
data_dir       = dc["data_dir"]        # ./sft_data
max_seq_length = dc["max_seq_length"]  # 2048

# ── checkpoints ──────────────────────────────────────────────────────────────
cc = cfg["checkpoint_config"]
base_model_path = cc["base_model_path"]   # pretrained weights
output_dir      = cc["output_dir"]        # ./sft_checkpoints
resume_from     = cc["resume_from"]       # None or path to SFT ckpt

# ── distributed ──────────────────────────────────────────────────────────────
distc = cfg["distributed_config"]
num_processes          = distc["num_processes"]          # 4
mixed_precision        = distc["mixed_precision"]        # bf16
find_unused_parameters = distc["find_unused_parameters"] # True (MoE experts)

# ── constants ────────────────────────────────────────────────────────────────
IGNORE_INDEX = -100   # label mask value written by sft_prepare.py

os.makedirs(output_dir, exist_ok=True)
print(json.dumps(cfg, indent=2))

{
  "model_config": {
    "vocab_size": 50304,
    "d_model": 384,
    "d_ff": 1024,
    "num_layers": 16,
    "num_experts": 16,
    "num_heads": 16,
    "block_size": 2048,
    "dropout": 0.1
  },
  "training_config": {
    "batch_size": 8,
    "gradient_accumulation_steps": 4,
    "max_iters": 50000,
    "eval_iters": 100,
    "log_interval": 2000,
    "save_interval": 5000,
    "lr": 1e-06,
    "min_lr": 1e-07,
    "warmup_iters": 500,
    "weight_decay": 0.01,
    "beta1": 0.95,
    "beta2": 0.99,
    "grad_clip": 1.0,
    "aux_loss_weight": 0.001
  },
  "data_config": {
    "data_dir": "./sft_data",
    "max_seq_length": 2048,
    "prompt_mask_fraction": 0.65,
    "train_split": 0.98,
    "val_split": 0.02,
    "dataset_mix": {
      "alpaca": 0,
      "dolly": 0,
      "evol_instruct": 20000,
      "everyday_conversations": 2200,
      "gsm8k": 8000,
      "code_python": 20000,
      "orca_math": 10000,
      "smollm_basics": 700,
      "smoltalk_10k": 15000
    },
    "n_eval_e

## 3 · Data loader

`sft_prepare.py` writes four flat binary files:

| file | dtype | meaning |
|---|---|---|
| `train_tokens.bin` | uint16 | token ids |
| `train_labels.bin` | int32  | token ids for response tokens, `-100` for prompt tokens |
| `val_tokens.bin`   | uint16 | — |
| `val_labels.bin`   | int32  | — |

We sample random windows of length `block_size` from the flat arrays.  
The label window is shifted by **0** (not +1) because `sft_prepare.py` already aligns tokens and labels at the same position — the model predicts token `t` using the label at position `t`.

In [4]:
def get_sft_batch(split: str, device, block_size: int, batch_size: int, data_dir: str):
    """
    Sample a random batch from the SFT binary files.

    Returns
    -------
    x : LongTensor  (batch_size, block_size)   — input token ids
    y : LongTensor  (batch_size, block_size)   — label ids (-100 = masked)

    Label alignment
    ---------------
    sft_prepare.py stores tokens and labels at the SAME index:
        tokens[i]  = token at position i
        labels[i]  = -100            if position i is a prompt token
                   = tokens[i]       if position i is a response token

    The model predicts the NEXT token, so we feed:
        x = tokens[i   : i+block_size]      (input)
        y = labels[i+1 : i+1+block_size]    (targets, shifted by 1)

    This means:
      • At every position t, the model sees x[t] and must predict y[t].
      • y[t] == -100  → prompt position → ignored by CrossEntropyLoss.
      • y[t] == token → response position → loss is computed.
    """
    token_path = os.path.join(data_dir, f"{split}_tokens.bin")
    label_path = os.path.join(data_dir, f"{split}_labels.bin")

    # Re-open memmap every call to avoid memory leaks (same pattern as pretraining)
    tokens = np.memmap(token_path, dtype=np.int32, mode="r")
    labels = np.memmap(label_path, dtype=np.int32,  mode="r")

    # Need at least block_size+1 tokens to form one (x, y) pair
    max_start = len(tokens) - block_size - 1
    ix = torch.randint(0, max_start, (batch_size,))

    x = torch.stack([
        torch.from_numpy(tokens[i: i + block_size].astype(np.int64)) for i in ix
    ])
    y = torch.stack([
        torch.from_numpy(labels[i+1: i+block_size+1].astype(np.int64)) for i in ix
    ])

    x = x.pin_memory().to(device, non_blocking=True)
    y = y.pin_memory().to(device, non_blocking=True)
    return x, y

### 3.1 · Quick sanity check on the data files

In [5]:
def verify_sft_data(data_dir: str, block_size: int) -> None:
    """
    Print basic statistics about the prepared SFT binary files and
    show one decoded example so you can visually confirm label masking.
    """
    import tiktoken
    #from sft_prepare_v2 import get_chatml_encoding
    enc = tiktoken.get_encoding("gpt2")

    for split in ("train", "val"):
        tok_path = os.path.join(data_dir, f"{split}_tokens.bin")
        lbl_path = os.path.join(data_dir, f"{split}_labels.bin")

        if not os.path.exists(tok_path):
            print(f"[WARN] {tok_path} not found — run sft_prepare.py first.")
            continue

        tokens = np.memmap(tok_path, dtype=np.int32, mode="r")
        labels = np.memmap(lbl_path, dtype=np.int32,  mode="r")

        supervised   = int((labels != IGNORE_INDEX).sum())
        masked       = int((labels == IGNORE_INDEX).sum())
        total        = len(tokens)
        mask_pct     = masked / max(total, 1) * 100

        print(f"\n{'='*60}")
        print(f"Split : {split}")
        print(f"  Total tokens      : {total:>12,}")
        print(f"  Supervised tokens : {supervised:>12,}  ({100-mask_pct:.1f}%)")
        print(f"  Masked tokens     : {masked:>12,}  ({mask_pct:.1f}%)")
        print(f"  Possible batches  : {(total - block_size - 1):>12,}")

    # ── Decode one window to visually verify masking ──────────────────────────
    tok_path = os.path.join(data_dir, "train_tokens.bin")
    lbl_path = os.path.join(data_dir, "train_labels.bin")
    if os.path.exists(tok_path):
        tokens = np.memmap(tok_path, dtype=np.int32, mode="r")
        labels = np.memmap(lbl_path, dtype=np.int32,  mode="r")

        # Find a window that contains at least one supervised token
        window = 512
        start  = 0
        for candidate in range(0, min(len(tokens) - window, 50_000), window):
            if (labels[candidate : candidate + window] != IGNORE_INDEX).any():
                start = candidate
                break

        tok_slice = tokens[start : start + window].astype(np.int64).tolist()
        lbl_slice = labels[start : start + window].astype(np.int64).tolist()

        print(f"\n{'='*60}")
        print(f"Sample window [tokens {start}:{start+window}]")
        print("\n[PROMPT — masked in labels]:")
        prompt_toks = [t for t, l in zip(tok_slice, lbl_slice) if l == IGNORE_INDEX]
        print(enc.decode(prompt_toks)[:1000])
        print("\n[RESPONSE — supervised in labels]:")
        resp_toks = [t for t, l in zip(tok_slice, lbl_slice) if l != IGNORE_INDEX]
        print(enc.decode(resp_toks)[:1000])

verify_sft_data(data_dir, block_size)


Split : train
  Total tokens      :   28,146,458
  Supervised tokens :   19,848,095  (70.5%)
  Masked tokens     :    8,298,363  (29.5%)
  Possible batches  :   28,144,409

Split : val
  Total tokens      :      574,418
  Supervised tokens :      406,034  (70.7%)
  Masked tokens     :      168,384  (29.3%)
  Possible batches  :      572,369

Sample window [tokens 0:512]

[PROMPT — masked in labels]:
### SYSTEM:
You are an instruction-following creative AI Assistant. Below is an instruction that describes a task. Write a response that appropriately completes the request

### Instruction:
In the movie "Blade Runner" Roy Batty's monologue at the end of the film touches on his own humanity and emotions and thus is able to touch the emotions of the viewer as in that moment he is very human like. How do we craft deep character moments like this into our own work?

### Response:


[RESPONSE — supervised in labels]:
The iconic monologue of Roy Batty - it's a masterclass in character developme

## 4 · Model + optimiser factory

Built outside the launcher so the objects can be inspected in the notebook before training starts.

In [6]:
model = MoETransformer(
    vocab_size  = vocab_size,
    d_model     = d_model,
    num_heads   = num_heads,
    d_ff        = d_ff,
    num_layers  = num_layers,
    num_experts = num_experts,
    max_seq_len = block_size,
    top_k       = 2,
    dropout     = dropout,
)

total_params     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
if cfg["training_config"].get("freeze_layers", 0) != 0:
        for idx in range(cfg["training_config"].get("freeze_layers", 0)):
            if idx < 0 or idx >= len(model.blocks):
                raise ValueError(f"Block index {idx} out of range (0-{len(model.blocks)-1})")
            for param in model.blocks[idx].parameters():
                param.requires_grad = False
            print(f"Froze layer {idx}")
else:
    print("No layers frozen")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters after freezing {cfg['training_config'].get('freeze_layers')} : {trainable_params:,}")
param_dict = {pn: p for pn, p in model.named_parameters()}
# filter out those that do not require grad
param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
# ── Separate weight-decay and no-decay parameter groups ──────────────────────
# Biases and LayerNorm parameters should NOT be weight-decayed.
decay_params     = [p for n, p in param_dict.items() if p.requires_grad and p.dim() >= 2]
no_decay_params  = [p for n, p in param_dict.items() if p.requires_grad and p.dim() < 2]

optimizer = AdamW(
    [
        {"params": decay_params,    "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr    = lr,
    betas = (beta1, beta2),
)

# ── LR schedule: linear warmup → cosine decay ────────────────────────────────
# warmup_iters steps of linear ramp from lr*0.1 → lr
# then cosine decay from lr → min_lr over the remaining steps
cosine_iters = max_iters - warmup_iters

warmup_scheduler = LinearLR(
    optimizer,
    start_factor = 0.1,
    end_factor   = 1.0,
    total_iters  = warmup_iters,
)
cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max   = cosine_iters,
    eta_min = min_lr,
)
scheduler = SequentialLR(
    optimizer,
    schedulers = [warmup_scheduler, cosine_scheduler],
    milestones = [warmup_iters],
)

print(f"Optimiser  : AdamW  lr={lr}  wd={weight_decay}  betas=({beta1},{beta2})")
print(f"Schedule   : LinearWarmup({warmup_iters} steps) → CosineDecay({cosine_iters} steps, min_lr={min_lr})")

Trainable parameters: 207728352
No layers frozen
Total parameters     : 207,728,352
Trainable parameters after freezing None : 207,728,352
Optimiser  : AdamW  lr=1e-06  wd=0.01  betas=(0.95,0.99)
Schedule   : LinearWarmup(500 steps) → CosineDecay(49500 steps, min_lr=1e-07)


## 5 · SFT training function

### Design notes

| Aspect | Pretraining (`train_moe.ipynb`) | SFT (this notebook) |
|---|---|---|
| Loss | `CrossEntropyLoss` on all tokens | `CrossEntropyLoss(ignore_index=-100)` — response tokens only |
| Data | `get_batch_from_rand_sample` (raw `.bin`) | `get_sft_batch` (token + label `.bin`) |
| Init | random | pretrained base model |
| LR | 5e-5 | 5e-6 (10× lower) |
| Schedule | cosine only | warmup + cosine |
| Aux loss | 1e-3 × MoE load-balance | same weight (`aux_loss_weight`) |

In [ ]:
def sft_training_fn(
    model,
    optimizer,
    scheduler,
    # ── hyper-params passed through notebook_launcher ──
    max_iters,
    log_interval,
    save_interval,
    eval_iters,
    grad_clip,
    aux_loss_weight,
    block_size,
    batch_size,
    data_dir,
    output_dir,
    base_model_path,
    resume_from,
    gradient_acc_steps,
    mixed_precision,
    find_unused_parameters,
):
    # ── Accelerator setup ─────────────────────────────────────────────────────
    ddp_kwargs  = DistributedDataParallelKwargs(find_unused_parameters=find_unused_parameters)
    accelerator = Accelerator(
        gradient_accumulation_steps = gradient_acc_steps,
        mixed_precision             = mixed_precision,
        kwargs_handlers             = [ddp_kwargs],
    )
    device = accelerator.device

    # ── Loss: ignore_index=-100 masks all prompt tokens ───────────────────────
    # This is the core SFT difference from pretraining.
    # sft_prepare.py sets label=-100 for every prompt/context token so the
    # model is never penalised for predicting them — only response tokens
    # contribute to the gradient.
    criterion = torch.nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

    # ── Determine starting iteration ─────────────────────────────────────────
    start_iter = 0

    if resume_from:
        # Resume a previous SFT run (not the pretrained base)
        accelerator.print(f"[Resume] Loading SFT checkpoint: {resume_from}")
        state = torch.load(resume_from, map_location="cpu", weights_only=True)
        model.load_state_dict(state["model"])
        #optimizer.load_state_dict(state["optimizer"])
        #scheduler.load_state_dict(state["scheduler"])
        start_iter = state.get("iter", 0)
        if start_iter > max_iters:
            start_iter = 0
            accelerator.print("[Warning] Resume checkpoint exceeds max_iters; starting from 0")
        accelerator.print(f"[Resume] Resuming from step {start_iter}")
    elif base_model_path:
        # Cold-start from pretrained base weights
        accelerator.print(f"[Init] Loading pretrained base: {base_model_path}")
        base_state = torch.load(base_model_path, map_location="cpu", weights_only=True)
        # The pretrained checkpoint may be a raw state_dict or wrapped dict
        if isinstance(base_state, dict) and "model" in base_state:
            base_state = base_state["model"]
        missing, unexpected = model.load_state_dict(base_state, strict=True)
        if missing:
            accelerator.print(f"[Init] Missing keys  : {missing}")
        if unexpected:
            accelerator.print(f"[Init] Unexpected keys: {unexpected}")
        accelerator.print("[Init] Pretrained weights loaded successfully.")
    # ── Prepare with Accelerate ───────────────────────────────────────────────
    model, optimizer, scheduler = accelerator.prepare(model, optimizer, scheduler)
    # ── Freeze embedding layer  ───────────────────────────────────────────────
    
    # ── Training loop ─────────────────────────────────────────────────────────
    model.train()
    optimizer.zero_grad()

    recent_loss            = 0.0
    recent_supervised_loss = 0.0   # loss on response tokens only (for logging)
    recent_steps           = 0

    gc.collect()
    torch.cuda.empty_cache()

    pbar = tqdm(
        total   = max_iters,
        initial = start_iter,
        desc    = "SFT",
        disable = not accelerator.is_local_main_process,
    )

    iter_num = start_iter
    total_supervised_tokens = 0
    while iter_num < max_iters:
        with accelerator.accumulate(model):
            # ── Forward pass ─────────────────────────────────────────────────
            X, y = get_sft_batch("train", device, block_size, batch_size, data_dir)

            logits, aux_loss = model(X)

            # ── SFT loss: only response tokens contribute ─────────────────────
            # logits : (B, T, vocab_size)
            # y      : (B, T)  with -100 for masked prompt positions
            supervised_loss = criterion(
                    logits.contiguous().view(-1, logits.size(-1)),
                    y.contiguous().view(-1),
            )
            num_supervised_tokens = (y != IGNORE_INDEX).sum().item()
            #num_supervised_tokens = accelerator.gather(torch.tensor((y != IGNORE_INDEX).sum(), device=device)
            #        ).sum().item()
            # ── MoE load-balancing auxiliary loss ─────────────────────────────
            loss = supervised_loss
            if aux_loss is not None:
                loss = loss + aux_loss_weight * aux_loss

            # ── Backward ─────────────────────────────────────────────────────
            accelerator.backward(loss)

            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

            optimizer.step()
            optimizer.zero_grad()
            # Scheduler steps once per *optimizer* step (not per micro-step)
            if accelerator.sync_gradients:
                scheduler.step()

            # ── Gather loss across all processes for logging ──────────────────
            gathered_loss  = accelerator.gather(loss.detach()).mean().item()
            gathered_sloss = accelerator.gather(supervised_loss.detach()).mean().item()
            recent_loss            += gathered_loss
            recent_supervised_loss += gathered_sloss
            recent_steps           += 1
            total_supervised_tokens += num_supervised_tokens
        
            if total_supervised_tokens >= max_supervised_tokens:
                print(f"Reached >max supervised tokens: {total_supervised_tokens}")
                break
        # ── Logging + validation ──────────────────────────────────────────────
        if (iter_num + 1) % log_interval == 0:
            avg_loss  = recent_loss            / max(recent_steps, 1)
            avg_sloss = recent_supervised_loss / max(recent_steps, 1)
            recent_loss = recent_supervised_loss = recent_steps = 0.0

            
            def estimate_val_loss():
                model.eval()
                with torch.no_grad():
                    val_losses = []
                    for _ in range(eval_iters):
                        Xv, yv = get_sft_batch("val", device, block_size, batch_size, data_dir)
                        lv, al = model(Xv)
                        vl = criterion(lv.contiguous().view(-1, lv.size(-1)), yv.contiguous().view(-1))
                        #if al is not None:
                        #    vl = vl + aux_loss_weight * al
                        val_losses.append(vl.detach())
                    mean_val =  torch.stack(val_losses).mean().item()
                model.train()
                return mean_val

            
            if accelerator.is_main_process:
                val_loss = estimate_val_loss()
                current_lr = optimizer.param_groups[0]["lr"]
                log_line = (
                    f"Step {iter_num+1:>6d} | "
                    f"train_loss={avg_loss:.6f} | "
                    f"supervised_loss={avg_sloss:.6f} | "
                    f"val_loss={val_loss:.6f} | "
                    f"lr={current_lr:.2e} | "
                    f"number of supervised tokens seen={total_supervised_tokens}"
                )
                print(log_line)
                with open(os.path.join(output_dir, "sft_training_log.txt"), "a") as flog:
                    flog.write(log_line + "\n")

        # ── Checkpoint saving ─────────────────────────────────────────────────
        if (iter_num + 1) % save_interval == 0 and accelerator.is_main_process:
            unwrapped = accelerator.unwrap_model(model)
            save_path = os.path.join(output_dir, f"sft_ckpt_full_{iter_num+1}.pt")
            # Save full training state so runs can be resumed cleanly
            accelerator.save(
                {
                    "model":     unwrapped.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "iter":      iter_num + 1,
                    "config":    cfg,
                },
                save_path,
            )
            accelerator.print(f"[Saved] {save_path}")

        iter_num += 1
        if accelerator.is_local_main_process:
            pbar.update(1)

    # ── Final checkpoint ──────────────────────────────────────────────────────
    if accelerator.is_main_process:
        pbar.close()
        unwrapped  = accelerator.unwrap_model(model)
        final_path = os.path.join(output_dir, "sft_ckpt_full_final.pt")
        accelerator.save(
            {
                "model":     unwrapped.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "iter":      iter_num,
                "config":    cfg,
            },
            final_path,
        )
        print(f"\nSFT training complete. Final checkpoint: {final_path}")

## 6 · Launch

All scalar hyper-parameters are passed explicitly through `args` so that each spawned process receives a clean copy — no reliance on notebook-level globals inside the training function.

In [ ]:
mp.set_start_method("spawn", force=True)

accelerate.notebook_launcher(
    sft_training_fn,
    args=(
        model,
        optimizer,
        scheduler,
        # ── scalars ──
        max_iters,
        log_interval,
        save_interval,
        eval_iters,
        grad_clip,
        aux_loss_weight,
        block_size,
        batch_size,
        data_dir,
        output_dir,
        base_model_path,
        resume_from,
        gradient_acc_steps,
        mixed_precision,
        find_unused_parameters,
    ),
    num_processes = num_processes,
    mixed_precision = mixed_precision,
)

Launching training on one GPU.
[Resume] Loading SFT checkpoint: ./sft_checkpoints/sft_ckpt_stage1c_34000.pt


[Resume] Resuming from step 34000


SFT:  68%|######8   | 34000/50000 [00:00<?, ?it/s]